In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset('CShorten/ML-ArXiv-Papers',split='train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df=df[['title','abstract']]

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df = df.head(15000)

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df['paper_text'] = df['title']+" "+df['abstract']

In [ ]:
df[['paper_text']].head()

In [ ]:
type(df[['paper_text']])

In [ ]:
print(df['paper_text'].iloc[0])

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
sample_text = df['paper_text'].iloc[0]
sample_text

In [ ]:
df['paper_text'] = df['paper_text'].astype(str)
df['paper_text'] = df['paper_text'].str.replace("\n", " ", regex=False)
df['paper_text'] = df['paper_text'].str.strip()

In [ ]:
import os
import numpy as np

if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")

    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)

    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

In [ ]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding = model.encode(df['paper_text'].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[-0].reshape(1,-1))
print(similarity)

In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)


# ***Generating Full Embedding***

In [ ]:
embedding = model.encode(df['paper_text'].to_list(),batch_size=32, show_progress_bar=True)


In [ ]:
print(embedding.shape)

In [ ]:
print(type(embedding))

In [ ]:
embedding.dtype

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
faiss.normalize_L2(embedding)

In [ ]:
index = faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding)

In [ ]:
print(index.ntotal)

In [ ]:
query = "deep leaning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
print(df.iloc[11873]["title"])

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
def search_paper(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
def search_query(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score:",score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()

In [ ]:
search_query("deep learning for medical image analysis")

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
type(summarizer)

In [ ]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
summary[0]

In [ ]:
for score,idx in zip(D[0],I[0]):
    print("Similarity Score:",score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]["summary_text"])
    print()


In [ ]:
def search_and_summarize(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score:",score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]["summary_text"])
    print()


In [ ]:
search_and_summarize("Deep learning in medical imaging", k=5)

In [ ]:
!pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT()

In [ ]:
type(kw_model) #model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
text = df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)

In [ ]:
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3), stop_words="english")

In [ ]:
print(keywords)

In [ ]:
def search_and_summarize(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score:",score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[idx]["summary_text"])
    print()
    keywords = kw_model.extract_keyword(text,keyphrase_ngram_range=(1,3),stop_words="english")
    print(keywords)
    for k,s in keywords:
      print(k)
      print(s)


In [ ]:
from transformers import pipeline
from keybert import KeyBERT

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")
kw_model = KeyBERT()

In [ ]:
def research_assistant(query, question, k=3):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        text = df.iloc[idx]["abstract"]
        title = df.iloc[idx]["title"]

        print(f"TITLE: {title} | SCORE: {score:.3f}\n")

        qa_result = qa_pipeline(question=question, context=text)
        if qa_result['score'] > 0.05:
            print(f"ANSWER: {qa_result['answer']}\n")

        summary = summarizer(text, max_length=120, min_length=40)
        print(f"SUMMARY: {summary[0]['summary_text']}\n")

        keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,2), stop_words="english")
        print("KEYWORDS:")
        for k_word, _ in keywords[:3]:
            print(f"- {k_word}")

        print("-" * 60)

In [ ]:
research_assistant(
    query="deep learning for medical image analysis",
    question="What specific type of neural network is used?",
    k=3
)